# 11 · Equal-budget clarification from documented episodes

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Requires actual fitting action episodes and available answers. Missing answers remain unknown; replay is not a real-user study.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Inspect actual action episodes
Each gain is based on independently reviewed before/after outputs. No gate-generated success labels.

In [ ]:
from oncoplate.clarification import GainPolicy,replay_focal_candidates,ACTIONS
import joblib
RUN_ID='dinov2_vits14_finetune_joint_s0'
action_root=p['private']/'clarification';action_root.mkdir(parents=True,exist_ok=True)
episodes=read_table(action_root/'fitting_action_episodes.csv')
for col in ['supported_gain','unsupported_gain','cost_units']:
    episodes[col]=episodes[col].astype(float)
print(episodes.groupby('action_id').size())
assert set(episodes.action_id)==set(ACTIONS)

## 2. Fit the proposed acquisition policy on fitting episodes only

In [ ]:
policy=GainPolicy(seed=0,penalty=1.,cost_weight=.05).fit(episodes)
# This locally produced joblib is trusted only within your own workspace; never load an unknown download.
joblib.dump(policy,action_root/'gain_policy_owned.joblib')
write_json(action_root/'action_protocol.json',{'budget':1,'penalty':1.,'cost_weight':.05,'actions':list(ACTIONS),'mode':'idealised_documented_answer_replay','fitting_only':True})

## 3. Generate matched one-action outputs for blind rating

In [ ]:
from oncoplate.inputs import load_context
from oncoplate.benchmark import make_rating_jobs
records,targets,states,rules=load_context(cfg,require_review=True)
context=read_json(p['private']/'protocol_context.json')
claim_root=p['private']/'claims'/RUN_ID/context['input_mode']
candidates=read_table(claim_root/'validation_candidates.csv');objects=read_jsonl(claim_root/'validation_objects.jsonl')
answers=read_jsonl(action_root/'documented_answers.jsonl')
for name,pol in [('fixed','fixed'),('random','random'),('uncertainty','uncertainty'),('pcsi',policy)]:
    after,oo=replay_focal_candidates(candidates,objects,answers,pol,rules,asof=context['asof'],budget=1,seed=0)
    write_table(action_root/f'{name}_validation_after_candidates.csv',after)
    write_jsonl(action_root/f'{name}_validation_after_objects.jsonl',oo)
    make_rating_jobs(after,action_root/f'{name}_validation_rating_jobs.csv')
    print(name, 'actions:',int(after.question_count.sum()), 'rating jobs:',len(after))

## 4. Report only after independent after-output ratings exist
Do not use immediate rule eligibility as the outcome.

In [ ]:
from oncoplate.benchmark import adjudicated_ratings,join_ratings
summaries=[]
for name in ['fixed','random','uncertainty','pcsi']:
    after=read_table(action_root/f'{name}_validation_after_candidates.csv')
    rated=adjudicated_ratings(read_table(action_root/f'{name}_validation_ratings_adjudicated.csv'))
    merged=join_ratings(after,rated)
    summaries.append({'policy':name,'record_count':len(merged),'supported_candidate_fraction':merged.support_status.eq('supported').mean(),
    'mean_actions':merged.question_count.astype(float).mean(),'mean_elapsed_seconds':merged.elapsed_seconds.astype(float).mean(),'condition':'idealised_documented_replay'})
import pandas as pd
write_table(p['reports']/'clarification_development_summary.csv',pd.DataFrame(summaries));display(pd.DataFrame(summaries))
print('These are development candidate-support summaries, not test selective-policy improvement or a human trial.')

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
